### ETL: bronze.air_quality_stations -> silver.meteorology_stations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, StringType
from delta.tables import DeltaTable 
import sys
import os
from pathlib import Path
current_dir = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
src_path = str(Path(current_dir).parents[1])
sys.path.append(src_path)

from utils.cleaning_functions import *

In [0]:
df_source = spark.table("dbw_routemind_euskadi_dev.bronze.air_quality_stations")

In [0]:
notnull_columns = [
    "sensorId",
    "latitude",
    "longitude",
    "municipality",
    "county"
]

keys=["sensorId", "category"]

In [0]:
df_exploded = explode_array_column(df_source,"features","feature")

In [0]:
df_extracted = df_exploded.select(
    col("feature.properties.id").alias("sensorId"),
    col("feature.geometry.coordinates")[1].cast(DoubleType()).alias("latitude"),
    col("feature.geometry.coordinates")[0].cast(DoubleType()).alias("longitude"),
    col("feature.properties.location.municipality").alias("municipality"),
    col("feature.properties.location.county").alias("county"),

)

In [0]:
df_clean = drop_null_required(df_extracted, notnull_columns)

In [0]:
df_final = add_category_column(df_clean, category_value="air_quality_station")

In [0]:
df_final = deduplicate(df_final, keys)

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.meteorology_stations"
delta_path = "abfss://silver@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/weather_stations/data"

if not spark.catalog.tableExists(target_table):
    print(f"Table {target_table} doesn't exist. Creating table...")
    
    df_final.write \
        .format("delta") \
        .option("path", delta_path) \
        .saveAsTable(target_table)
        
    print(f"table {target_table} created. rows processed: {df_final.count()}")

else:
    delta_target = DeltaTable.forName(spark, target_table)
    (
        delta_target.alias("t")
        .merge(
            df_final.alias("s"),
            "t.sensorId = s.sensorId AND t.category = s.category"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE completed on {target_table}. rows processed: {df_final.count()}")